# Day 1, step 1: is there a usable timestamp?

Before any time-based split, confirm Dreaddit has a real date for each post. Run `python src/download_data.py` first.

In [1]:
import pandas as pd

train = pd.read_csv("../data/raw/dreaddit_train.csv")
test = pd.read_csv("../data/raw/dreaddit_test.csv")
print(train.shape, test.shape)

# 116 columns, but most are pre-computed features (lex_liwc_*, lex_dal_*, syntax_*).
# We only care about the ones that describe the post itself.
core = ["subreddit", "post_id", "sentence_range", "text", "label", "confidence", "social_timestamp"]
train[core].head()

(2838, 116) (715, 116)


,subreddit,post_id,sentence_range,text,label,confidence,social_timestamp
0,ptsd,8601tu,"(15, 20)","He said he had not felt that way before, sugge...",1,0.8,1521614353
1,assistance,8lbrx9,"(0, 5)","Hey there r/assistance, Not sure if this is th...",0,1.0,1527009817
2,ptsd,9ch1zh,"(15, 20)",My mom then hit me with the newspaper and it s...,1,0.8,1535935605
3,relationships,7rorpp,"[5, 10]","until i met my new boyfriend, he is amazing, h...",1,0.6,1516429555
4,survivorsofabuse,9p2gbc,"[0, 5]",October is Domestic Violence Awareness Month a...,1,0.8,1539809005


In [2]:
# social_timestamp is Unix time: seconds since 1970-01-01 UTC.
for name, df in [("train", train), ("test", test)]:
    ts = pd.to_datetime(df["social_timestamp"], unit="s", utc=True)
    print(f"{name}: missing={ts.isna().sum()}  earliest={ts.min()}  latest={ts.max()}")

train: missing=0  earliest=2017-01-01 12:36:58+00:00  latest=2018-11-19 01:54:20+00:00
test: missing=0  earliest=2017-01-02 04:02:57+00:00  latest=2018-11-18 23:20:24+00:00


In [3]:
# A temporal split needs posts spread across time, not bunched into one week.
train["month"] = pd.to_datetime(train["social_timestamp"], unit="s", utc=True).dt.strftime("%Y-%m")
train.groupby("month").agg(rows=("label", "size"), stress_rate=("label", "mean"))

C:\Users\khana\AppData\Local\Temp\ipykernel_67196\1131746077.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["month"] = pd.to_datetime(train["social_timestamp"], unit="s", utc=True).dt.strftime("%Y-%m")


,rows,stress_rate
month,,
2017-01,88,0.670455
2017-02,46,0.565217
2017-03,90,0.488889
2017-04,51,0.431373
2017-05,60,0.600000
2017-06,63,0.587302
2017-07,76,0.592105
2017-08,67,0.641791
2017-09,69,0.550725


In [4]:
# One Reddit post can appear as several rows (different sentence_range chunks).
# Check that all chunks of a post share one timestamp, so a time split can't cut a post in half.
per_post = train.groupby("post_id")["social_timestamp"].nunique()
print("unique posts:", len(per_post), " posts with >1 row:", (train["post_id"].value_counts() > 1).sum())
print("posts whose rows have different timestamps:", (per_post > 1).sum())

unique posts: 2343  posts with >1 row: 380
posts whose rows have different timestamps: 0


# Day 1, step 2: what is the January 2018 spike?

January 2018 has ~650 train rows; most months have 50–180. Before choosing a cutoff date for Split B, find out what is in that month.

In [5]:
# Which subreddits make up the spike?
jan18 = train[train["month"] == "2018-01"]
print("Jan 2018 rows:", len(jan18), "unique posts:", jan18["post_id"].nunique())
jan18["subreddit"].value_counts()

Jan 2018 rows: 646 unique posts: 497


subreddit
relationships       552
anxiety              27
ptsd                 19
homeless             13
assistance           12
domesticviolence      8
survivorsofabuse      7
food_pantry           4
stress                2
almosthomeless        2
Name: count, dtype: int64

In [6]:
# Ruling out "one busy day": if a single event caused the spike, most rows would sit on 1-2 days.
# .dt.date drops the time of day, so value_counts() gives rows per calendar day (busiest first).
jan18_days = pd.to_datetime(jan18["social_timestamp"], unit="s", utc=True).dt.date
print("days in Jan 2018 with at least one row:", jan18_days.nunique())
jan18_days.value_counts().head(10)

days in Jan 2018 with at least one row: 31


social_timestamp
2018-01-23    36
2018-01-04    35
2018-01-28    34
2018-01-05    33
2018-01-29    29
2018-01-07    26
2018-01-13    25
2018-01-18    24
2018-01-30    22
2018-01-24    22
Name: count, dtype: int64

In [7]:
# Is that subreddit present in other months too? Count rows per subreddit per year, train and test.
# crosstab = a count table: one row per subreddit, one column per year.
for name, df in [("train", train), ("test", test)]:
    year = pd.to_datetime(df["social_timestamp"], unit="s", utc=True).dt.year
    print(f"--- {name} ---")
    print(pd.crosstab(df["subreddit"], year), "\n")

--- train ---
social_timestamp  2017  2018
subreddit                   
almosthomeless      42    38
anxiety            199   304
assistance         119   170
domesticviolence   130   186
food_pantry          9    28
homeless            42   126
ptsd               218   366
relationships        0   552
stress              18    46
survivorsofabuse    95   150 

--- test ---
social_timestamp  2017  2018
subreddit                   
almosthomeless      12     7
anxiety             76    71
assistance          21    45
domesticviolence    28    44
food_pantry          3     3
homeless            18    34
ptsd                39    88
relationships        0   142
stress               4    10
survivorsofabuse    27    43 



Can see that the subreddit r/relationships appear only in 2018 in both train and test. Check to see if that is the reason for the Jan 2018 spike.

In [8]:
# Where in time does r/relationships appear at all?
for name, df in [("train", train), ("test", test)]:
    rel_dates = pd.to_datetime(df.loc[df["subreddit"] == "relationships", "social_timestamp"], unit="s", utc=True)
    print(f"--- {name} ---")
    print(f"r/relationships ({name}):", rel_dates.min(), "to", rel_dates.max(), "\n")


--- train ---
r/relationships (train): 2018-01-01 05:36:27+00:00 to 2018-01-31 04:40:28+00:00 

--- test ---
r/relationships (test): 2018-01-01 07:02:39+00:00 to 2018-01-31 04:39:12+00:00 



# Day 1, Step 4: Choosing the Split B cutoff

For the Split B (by date), we will look to find the date which will give us the 80/20 split on the train and test data.

In [9]:
# For the temporal split, we want to avoid cutting a post in half. All chunks of a post share one timestamp (step 1), so cutting by date keeps them together.

clean = pd.read_csv("../data/processed/dreaddit_clean.csv")
print(clean.shape)
print(list(clean.columns))

# sentence_range was not kept by prepare_data.py, so build the list from columns that exist.
core = ["subreddit", "post_id", "text", "label", "confidence", "social_timestamp", "orig_split"]
clean[core].head()

(2859, 8)
['subreddit', 'post_id', 'text', 'label', 'confidence', 'social_timestamp', 'orig_split', 'date']


,subreddit,post_id,text,label,confidence,social_timestamp,orig_split
0,domesticviolence,5lezhz,He came after me a few times so I got away and...,1,0.571429,1483274218,train
1,domesticviolence,5lezhz,"We will get through this and I am determined, ...",1,0.857143,1483274218,train
2,anxiety,5lge1y,Currently in s constant state of anxiety with ...,1,1.000000,1483295706,train
3,assistance,5lj7sy,All the research are being conducted by Harvar...,0,1.000000,1483329777,test
4,anxiety,5lkwas,"Well maybe afraid isn't the best word, but it'...",1,1.000000,1483357819,test


In [10]:
# quantile(0.8) = the timestamp that has 80% of rows at or before it.
cutoff = clean["social_timestamp"].quantile(0.8)
print("cutoff:", pd.to_datetime(cutoff, unit="s", utc=True))

# True = older than the cutoff -> train; False -> test. ~ flips True/False.
is_train = clean["social_timestamp"] < cutoff
b_train, b_test = clean[is_train], clean[~is_train]
print(f"train rows: {len(b_train)}   test rows: {len(b_test)}   test share: {(~is_train).mean():.3f}")

cutoff: 2018-08-31 04:41:56.599999905+00:00
train rows: 2287   test rows: 572   test share: 0.200


In [11]:
# Check 1: no post on both sides (should be 0).
print("posts on both sides:", len(set(b_train["post_id"]) & set(b_test["post_id"])))

# Check 2: class balance. Compare with Split A (0.543 train / 0.539 test).
print(f"stress rate   train: {b_train['label'].mean():.3f}   test: {b_test['label'].mean():.3f}")

# Date range of each side, to see what "older" and "newer" actually mean.
for name, part in [("train", b_train), ("test", b_test)]:
    d = pd.to_datetime(part["social_timestamp"], unit="s", utc=True)
    print(f"{name} dates: {d.min().date()} to {d.max().date()}")

posts on both sides: 0
stress rate   train: 0.539   test: 0.556
train dates: 2017-01-01 to 2018-08-31
test dates: 2018-08-31 to 2018-11-19


In [12]:
# Check 3: subreddit mix. If one side is mostly a different subreddit, the split becomes a topic split again.
# normalize="columns" turns counts into shares, so each column adds up to 1.
side = is_train.map({True: "train", False: "test"})
mix = pd.crosstab(clean["subreddit"], side, normalize="columns").round(3)
mix["diff"] = (mix["test"] - mix["train"]).round(3)
mix.sort_values("diff")

social_timestamp,test,train,diff
subreddit,,,
anxiety,0.205,0.233,-0.028
domesticviolence,0.119,0.140,-0.021
almosthomeless,0.021,0.038,-0.017
survivorsofabuse,0.108,0.111,-0.003
homeless,0.075,0.077,-0.002
food_pantry,0.019,0.014,0.005
stress,0.037,0.025,0.012
assistance,0.145,0.119,0.026
ptsd,0.271,0.243,0.028


**Decision:** the 80% quantile lands at 2018-08-31 04:41 UTC. We fix the cutoff at **2018-09-01 00:00 UTC** instead. It is easier to explain (train Jan 2017–Aug 2018, test Sep–Nov 2018) and doesn't move if the data processing changes. It shifts 7 rows into train: 2,294 train / 565 test (test share 0.198), stress rate 0.538 / 0.558, 0 posts on both sides. Implemented in `src/splits.py`.